In [ ]:

##Notebook 2 — Nettoyage et transformation des données
## 💡 Créer un notebook 02_nettoyage.ipynb. Ce jalon couvre le nettoyage et les transformations colonne par colonne.


In [25]:
## Initialisation session Spark et chargement des données
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, upper, round, year,concat, lit
from pyspark.sql.types import DateType, DoubleType, IntegerType


# Initialisation de la session Sparks
spark = SparkSession.builder \
    .appName("TradeCorp_Nettoyage") \
    .getOrCreate()

path = "/home/jovyan/data/raw/"

# Chargement des 8 fichiers CSV bruts
df_categories = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}categories.csv")
df_customers = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}customers.csv")
df_employees = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}employees.csv")
df_order_details = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}order_details.csv")
df_orders = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}orders.csv")
df_products = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}products.csv")
df_shippers = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}shippers.csv")
df_suppliers = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}suppliers.csv")

print("Chargement des 8 tables effectué avec succès !")

Chargement des 8 tables effectué avec succès !


In [26]:
## Q11 — Valeurs nulles - Pour chaque DataFrame, compter le nombre de valeurs nulles par colonne. Utiliser une boucle et df.filter(col(c).isNull()).count().

dfs = {
    "Categories": df_categories,
    "Customers": df_customers,
    "Employees": df_employees,
    "Order Details": df_order_details,
    "Orders": df_orders,
    "Products": df_products,
    "Shippers": df_shippers,
    "Suppliers": df_suppliers
}

for name, df in dfs.items():
    print(f"=== Valeurs nulles pour {name} ===")
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        print(f"Colonne {c} : {null_count} valeurs nulles")
    print("\n")


=== Valeurs nulles pour Categories ===
Colonne category_id : 0 valeurs nulles
Colonne category_name : 0 valeurs nulles
Colonne description : 0 valeurs nulles
Colonne picture : 8 valeurs nulles


=== Valeurs nulles pour Customers ===
Colonne customer_id : 0 valeurs nulles
Colonne company_name : 0 valeurs nulles
Colonne contact_name : 0 valeurs nulles
Colonne contact_title : 0 valeurs nulles
Colonne address : 0 valeurs nulles
Colonne city : 0 valeurs nulles
Colonne region : 60 valeurs nulles
Colonne postal_code : 1 valeurs nulles
Colonne country : 0 valeurs nulles
Colonne phone : 0 valeurs nulles
Colonne fax : 22 valeurs nulles


=== Valeurs nulles pour Employees ===
Colonne employee_id : 0 valeurs nulles
Colonne last_name : 0 valeurs nulles
Colonne first_name : 0 valeurs nulles
Colonne title : 0 valeurs nulles
Colonne title_of_courtesy : 0 valeurs nulles
Colonne birth_date : 0 valeurs nulles
Colonne hire_date : 0 valeurs nulles
Colonne address : 0 valeurs nulles
Colonne city : 0 valeurs

In [27]:
## Q12 — Supprimer les nulls - Dans df_orders, supprimer les lignes où shipped_date est null (commandes non livrées). Dans df_products, remplacer les valeurs nulles de unit_price par la médiane.

# 1. Dans df_orders, supprimer les lignes où shipped_date est null
df_orders = df_orders.filter(col("shipped_date").isNotNull())

# 2. Dans df_products, remplacer les valeurs nulles de unit_price par la médiane
# Calcul de la médiane (approxQuantile retourne une liste, on prend le premier élément [0])
median_price = df_products.approxQuantile("unit_price", [0.5], 0.0)[0]

# Remplacement de la valeur nulle par la médiane
df_products = df_products.na.fill({"unit_price": median_price})

print(f"Lignes non livrées supprimées dans df_orders.")
print(f"Médiane de unit_price appliquée dans df_products : {median_price}")

Lignes non livrées supprimées dans df_orders.
Médiane de unit_price appliquée dans df_products : 19.5


In [10]:
## Verifications

# 1. Vérifier qu'il ne reste plus aucun null dans shipped_date pour df_orders
null_shipped_orders = df_orders.filter(col("shipped_date").isNull()).count()
print(f"Nombre de valeurs nulles restantes dans 'shipped_date' (df_orders) : {null_shipped_orders}")

# 2. Vérifier qu'il ne reste plus aucun null dans unit_price pour df_products
null_price_products = df_products.filter(col("unit_price").isNull()).count()
print(f"Nombre de valeurs nulles restantes dans 'unit_price' (df_products) : {null_price_products}")


Nombre de valeurs nulles restantes dans 'shipped_date' (df_orders) : 0
Nombre de valeurs nulles restantes dans 'unit_price' (df_products) : 0


In [28]:
##Q13 — Cast des types - Dans df_orders, caster order_date, required_date et shipped_date en type DateType. Dans df_order_details, caster unit_price en DoubleType et quantity en IntegerType

# 1. Dans df_orders, caster les colonnes de dates en DateType
df_orders = df_orders \
    .withColumn("order_date", col("order_date").cast(DateType())) \
    .withColumn("required_date", col("required_date").cast(DateType())) \
    .withColumn("shipped_date", col("shipped_date").cast(DateType()))

# 2. Dans df_order_details, caster unit_price en DoubleType et quantity en IntegerType
df_order_details = df_order_details \
    .withColumn("unit_price", col("unit_price").cast(DoubleType())) \
    .withColumn("quantity", col("quantity").cast(IntegerType()))

print("Casts de types effectués avec succès pour df_orders et df_order_details !")
## Verifications
print("--- Schéma de df_orders ---")
df_orders.printSchema()

print("\n--- Schéma de df_order_details ---")
df_order_details.printSchema()



Casts de types effectués avec succès pour df_orders et df_order_details !
--- Schéma de df_orders ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)


--- Schéma de df_order_details ---
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



In [29]:
##Q14 — Nettoyage des chaînes - Dans df_customers, appliquer TRIM sur toutes les colonnes texte. Mettre contact_name en title case avec initcap(). Mettre country en majuscules avec upper().

# 1. Appliquer TRIM sur toutes les colonnes de type string (texte) dans df_customers
for c, dtype in df_customers.dtypes:
    if dtype == "string":
        df_customers = df_customers.withColumn(c, trim(col(c)))

# 2. Mettre contact_name en title case et country en majuscules
df_customers = df_customers \
    .withColumn("contact_name", initcap(col("contact_name"))) \
    .withColumn("country", upper(col("country")))

print("Nettoyage des chaînes de caractères effectué avec succès dans df_customers !")

## Verifications
# Afficher un aperçu des colonnes modifiées sur quelques lignes
print("Vérification des transformations sur df_customers (5 premières lignes) :")

df_customers.select(
    "customer_id",   # Pour voir l'ID d'origine
    "company_name",  # Pour vérifier le TRIM (pas d'espace au début/fin)
    "contact_name",  # Pour vérifier le Title Case (ex: 'Maria Anders')
    "country"        # Pour vérifier les majuscules (ex: 'FRANCE')
).show(5, truncate=False)

Nettoyage des chaînes de caractères effectué avec succès dans df_customers !
Vérification des transformations sur df_customers (5 premières lignes) :
+-----------+----------------------------------+------------------+-------+
|customer_id|company_name                      |contact_name      |country|
+-----------+----------------------------------+------------------+-------+
|ALFKI      |Alfreds Futterkiste               |Maria Anders      |GERMANY|
|ANATR      |Ana Trujillo Emparedados y helados|Ana Trujillo      |MEXICO |
|ANTON      |Antonio Moreno Taquería           |Antonio Moreno    |MEXICO |
|AROUT      |Around the Horn                   |Thomas Hardy      |UK     |
|BERGS      |Berglunds snabbköp                |Christina Berglund|SWEDEN |
+-----------+----------------------------------+------------------+-------+
only showing top 5 rows



In [30]:
## Q15 — Renommer les colonnes - Dans df_order_details, renommer unit_price en prix_unitaire et quantity en quantite. Dans df_orders, renommer ship_via en shipper_id.

# 1. Dans df_order_details, renommer unit_price en prix_unitaire et quantity en quantite
df_order_details = df_order_details \
    .withColumnRenamed("unit_price", "prix_unitaire") \
    .withColumnRenamed("quantity", "quantite")

# 2. Dans df_orders, renommer ship_via en shipper_id
df_orders = df_orders \
    .withColumnRenamed("ship_via", "shipper_id")

print("Renommage des colonnes effectué avec succès !")

## Vérifications
print("--- Vérification du schéma de df_order_details ---")
df_order_details.printSchema()

print("\n--- Vérification du schéma de df_orders ---")
df_orders.printSchema()

Renommage des colonnes effectué avec succès !
--- Vérification du schéma de df_order_details ---
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)


--- Vérification du schéma de df_orders ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)



In [31]:
## Q16 — Colonnes calculées - Dans df_order_details, ajouter une colonne sous_total = prix_unitaire * quantite * (1 - discount). Arrondir à 2 décimales avec round().

# Ajout de la colonne sous_total arrondie à 2 décimales
df_order_details = df_order_details.withColumn(
    "sous_total", 
    round(col("prix_unitaire") * col("quantite") * (1 - col("discount")), 2)
)

print("Colonne 'sous_total' ajoutée avec succès dans df_order_details !")
df_order_details.select("order_id", "product_id", "prix_unitaire", "quantite", "discount", "sous_total").show(5)



Colonne 'sous_total' ajoutée avec succès dans df_order_details !
+--------+----------+-------------+--------+--------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|
+--------+----------+-------------+--------+--------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|
|   10248|        42|          9.8|      10|     0.0|      98.0|
|   10248|        72|         34.8|       5|     0.0|     174.0|
|   10249|        14|         18.6|       9|     0.0|     167.4|
|   10249|        51|         42.4|      40|     0.0|    1696.0|
+--------+----------+-------------+--------+--------+----------+
only showing top 5 rows



In [32]:
## Q17 — Colonnes conditionnelles - Dans df_products, ajouter une colonne en_stock (True si units_in_stock > 0). Dans df_orders, ajouter une colonne is_shipped (True si shipped_date n'est pas null).

# 1. Dans df_products, ajouter en_stock (True si units_in_stock > 0)
df_products = df_products.withColumn(
    "en_stock", 
    col("units_in_stock") > 0
)

# 2. Dans df_orders, ajouter is_shipped (True si shipped_date n'est pas null)
# Note: On a déjà filtré les nulls à la Q12, mais on applique la règle conditionnelle demandée
df_orders = df_orders.withColumn(
    "is_shipped", 
    col("shipped_date").isNotNull()
)

print("Colonnes conditionnelles 'en_stock' et 'is_shipped' ajoutées avec succès !")

## Verifications
print("--- Vérification df_products ---")
df_products.select("product_id", "product_name", "units_in_stock", "en_stock").show(5)

print("--- Vérification df_orders ---")
df_orders.select("order_id", "shipped_date", "is_shipped").show(5)

Colonnes conditionnelles 'en_stock' et 'is_shipped' ajoutées avec succès !
--- Vérification df_products ---
+----------+--------------------+--------------+--------+
|product_id|        product_name|units_in_stock|en_stock|
+----------+--------------------+--------------+--------+
|         1|                Chai|            39|    true|
|         2|               Chang|            17|    true|
|         3|       Aniseed Syrup|            13|    true|
|         4|Chef Anton's Caju...|            53|    true|
|         5|Chef Anton's Gumb...|             0|   false|
+----------+--------------------+--------------+--------+
only showing top 5 rows

--- Vérification df_orders ---
+--------+------------+----------+
|order_id|shipped_date|is_shipped|
+--------+------------+----------+
|   10248|  1996-07-16|      true|
|   10249|  1996-07-10|      true|
|   10250|  1996-07-12|      true|
|   10251|  1996-07-15|      true|
|   10252|  1996-07-11|      true|
+--------+------------+----------+

In [33]:
## Q18 — Doublons - Vérifier s'il y a des doublons dans df_customers sur customer_id. Utiliser distinct() et count() pour comparer. Supprimer les doublons si nécessaire.

# 1. Comparer le nombre total de lignes et le nombre de customer_id uniques
total_rows = df_customers.count()
unique_customers = df_customers.select("customer_id").distinct().count()

print(f"Nombre total de lignes : {total_rows}")
print(f"Nombre de customer_id uniques : {unique_customers}")

# 2. Supprimer les doublons sur customer_id (ou sur toute la ligne)
df_customers = df_customers.dropDuplicates(["customer_id"])

print("Suppression des doublons effectuée avec succès !")



Nombre total de lignes : 91
Nombre de customer_id uniques : 91
Suppression des doublons effectuée avec succès !


In [34]:
## Q19 — Filtrage - Filtrer df_orders pour ne garder que les commandes de 1997. Filtrer df_products pour ne garder que les produits en stock (units_in_stock > 0) et non discontinués.

# 1. Filtrer df_orders pour ne garder que les commandes de l'année 1997
df_orders = df_orders.filter(year(col("order_date")) == 1997)

# 2. Filtrer df_products pour ne garder que les produits en stock (units_in_stock > 0) ET non discontinués (discontinued == 0)
# Note : on utilise la colonne "en_stock" qu'on a créée à la Q17 ou directement "units_in_stock > 0"
df_products = df_products.filter((col("units_in_stock") > 0) & (col("discontinued") == 0))

print("Filtrages effectués avec succès !")

## Verifications
print(f"Nombre de commandes en 1997 : {df_orders.count()}")
print(f"Nombre de produits en stock et non discontinués : {df_products.count()}")

print("\nAperçu des commandes 1997 :")
df_orders.select("order_id", "order_date").show(5)

print("\nAperçu des produits filtrés :")
df_products.select("product_id", "product_name", "units_in_stock", "discontinued").show(5)

Filtrages effectués avec succès !
Nombre de commandes en 1997 : 408
Nombre de produits en stock et non discontinués : 66

Aperçu des commandes 1997 :
+--------+----------+
|order_id|order_date|
+--------+----------+
|   10400|1997-01-01|
|   10401|1997-01-01|
|   10402|1997-01-02|
|   10403|1997-01-03|
|   10404|1997-01-03|
+--------+----------+
only showing top 5 rows


Aperçu des produits filtrés :
+----------+--------------------+--------------+------------+
|product_id|        product_name|units_in_stock|discontinued|
+----------+--------------------+--------------+------------+
|         3|       Aniseed Syrup|            13|           0|
|         4|Chef Anton's Caju...|            53|           0|
|         6|Grandma's Boysenb...|           120|           0|
|         7|Uncle Bob's Organ...|            15|           0|
|         8|Northwoods Cranbe...|             6|           0|
+----------+--------------------+--------------+------------+
only showing top 5 rows



In [35]:
## Q20 — Sélection de colonnes - Dans df_employees, sélectionner uniquement : employee_id, first_name, last_name, title, hire_date, city, country. Créer une colonne full_name = first_name + ' ' + last_name

# Sélection des colonnes demandées et création de full_name
df_employees = df_employees.select(
    "employee_id",
    "first_name",
    "last_name",
    concat(col("first_name"), lit(" "), col("last_name")).alias("full_name"),
    "title",
    "hire_date",
    "city",
    "country"
)

print("Sélection et création de 'full_name' effectuées avec succès dans df_employees !")

## Verif
df_employees.select("employee_id", "full_name", "title", "country").show(5, truncate=False)


Sélection et création de 'full_name' effectuées avec succès dans df_employees !
+-----------+----------------+---------------------+-------+
|employee_id|full_name       |title                |country|
+-----------+----------------+---------------------+-------+
|1          |Nancy Davolio   |Sales Representative |USA    |
|2          |Andrew Fuller   |Vice President, Sales|USA    |
|3          |Janet Leverling |Sales Representative |USA    |
|4          |Margaret Peacock|Sales Representative |USA    |
|5          |Steven Buchanan |Sales Manager        |UK     |
+-----------+----------------+---------------------+-------+
only showing top 5 rows



In [37]:
## Écrire les DataFrames nettoyés en Parquet : PATH = "/home/jovyan/data/tmp"

PATH = "/home/jovyan/data/tmp"

# 1. Vérification / Création du dossier cible
os.makedirs(PATH, exist_ok=True)
print(f"Dossier validé ou créé : {PATH}\n")

# 2. Sauvegarde de tous les DataFrames
datasets = {
    "customers": df_customers,
    "products": df_products,
    "orders": df_orders,
    "order_details": df_order_details,
    "employees": df_employees
}

for name, df in datasets.items():
    folder_path = os.path.join(PATH, name)
    df.write.mode("overwrite").parquet(folder_path)
    print(f"[{name}] -> Sauvegardé avec succès.")

print("\nToutes les écritures sont terminées !")

## Verifications
!ls -la /home/jovyan/data/tmp


Dossier validé ou créé : /home/jovyan/data/tmp

[customers] -> Sauvegardé avec succès.
[products] -> Sauvegardé avec succès.
[orders] -> Sauvegardé avec succès.
[order_details] -> Sauvegardé avec succès.
[employees] -> Sauvegardé avec succès.

Toutes les écritures sont terminées !
total 28
drwxr-xr-x 7 jovyan users 4096 Aug 29 16:40 .
drwxrwxrwx 5 jovyan  1000 4096 Aug 29 16:34 ..
drwxr-xr-x 2 jovyan users 4096 Aug 29 16:40 customers
drwxr-xr-x 2 jovyan users 4096 Aug 29 16:40 employees
drwxr-xr-x 2 jovyan users 4096 Aug 29 16:40 order_details
drwxr-xr-x 2 jovyan users 4096 Aug 29 16:40 orders
drwxr-xr-x 2 jovyan users 4096 Aug 29 16:40 products


SyntaxError: invalid syntax (1328956080.py, line 1)